In [ ]:
%pip install numpy pandas scipy scikit-learn

In [16]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds
from sklearn.preprocessing import MinMaxScaler
from collections import defaultdict, Counter
from functools import partial


In [ ]:
# bước chuẩn bị dữ liệu
%run Prepare_data.ipynb

# Xây dựng mô hình Hybrid ( SVD + Content-Based )

In [17]:
def precision_at_k(recommended, actual, k=10):
    if not recommended:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    hits = len(set(recommended_k) & actual_set)
    return hits / k

def recall_at_k(recommended, actual, k=10):
    if not actual:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    hits = len(set(recommended_k) & actual_set)
    return hits / len(actual_set)

def ndcg_at_k(recommended, actual, k=10):
    if not recommended:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    dcg = 0.0
    for i, item in enumerate(recommended_k):
        if item in actual_set:
            dcg += 1.0 / np.log2(i + 2)
    ideal_hits = min(len(actual_set), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

In [18]:
def build_user_ground_truth(test_df):
    """
    Xây dựng ground truth cho bài toán user-to-item.
    Parameters:
        test_df: DataFrame chứa 'user_id', 'parent_asin', 'liked'
    Returns:
        dict: user_id -> set of parent_asin (các item liked trong test)
    """
    user_gt = test_df[test_df['liked'] == 1].groupby('user_id')['parent_asin'].apply(set).to_dict()
    return user_gt

def build_item_ground_truth(train_df, top_m=20):
    """
    Xây dựng ground truth cho bài toán item-to-item dựa trên co-occurrence.
    Parameters:
        train_df: DataFrame chứa 'user_id', 'parent_asin', 'liked' (chỉ lấy liked=1)
        top_m: số lượng item liên quan tối đa cho mỗi item (dùng để giới hạn ground truth)
    Returns:
        dict: parent_asin -> set of parent_asin (các item liên quan)
    """
    # Chỉ xét các tương tác liked=1
    liked_df = train_df[train_df['liked'] == 1]
    # Nhóm các item theo user
    user_items = liked_df.groupby('user_id')['parent_asin'].apply(set).to_dict()
    
    # Đếm số lần hai item cùng xuất hiện
    cooccur = defaultdict(Counter)
    for items in user_items.values():
        items_list = list(items)
        for i in range(len(items_list)):
            for j in range(i+1, len(items_list)):
                a, b = items_list[i], items_list[j]
                cooccur[a][b] += 1
                cooccur[b][a] += 1
    
    # Với mỗi item, lấy top_m item có số lần cùng xuất hiện cao nhất
    item_gt = {}
    for item, counter in cooccur.items():
        # Lấy top_m item (có thể ít hơn nếu không đủ)
        top_items = [it for it, _ in counter.most_common(top_m)]
        item_gt[item] = set(top_items)
    
    return item_gt

In [19]:
def evaluate_user_to_item(train_df, test_df, recommend_func, k=10):
    """
    Đánh giá mô hình user-to-item.
    """
    # Sử dụng hàm ground truth đã định nghĩa
    user_actual = build_user_ground_truth(test_df)  # thay vì viết trực tiếp
    user_interacted = train_df.groupby('user_id')['parent_asin'].apply(set).to_dict()
    
    precisions = []
    recalls = []
    ndcgs = []
    
    for user, actual in user_actual.items():
        if not actual:
            continue
        interacted = user_interacted.get(user, set())
        recommended = recommend_func(user, interacted, k)
        recommended = [item for item in recommended if item not in interacted]
        precisions.append(precision_at_k(recommended, actual, k))
        recalls.append(recall_at_k(recommended, actual, k))
        ndcgs.append(ndcg_at_k(recommended, actual, k))
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'ndcg': np.mean(ndcgs)
    }

In [20]:
def evaluate_item_to_item(train_df, item_ground_truth, recommend_func_item, k=10):
    """
    Đánh giá mô hình item-to-item.
    
    Parameters:
        train_df: không dùng trực tiếp, nhưng giữ để đồng bộ
        item_ground_truth: dict {item: set of related items}
        recommend_func_item: hàm func(item_id, k) -> list
        k: số lượng gợi ý
    """
    precisions = []
    recalls = []
    ndcgs = []
    
    for item, actual in item_ground_truth.items():
        if not actual:
            continue
        recommended = recommend_func_item(item, k)
        p = precision_at_k(recommended, actual, k)
        r = recall_at_k(recommended, actual, k)
        n = ndcg_at_k(recommended, actual, k)
        precisions.append(p)
        recalls.append(r)
        ndcgs.append(n)
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'ndcg': np.mean(ndcgs)
    }

In [25]:
# ==============================================
# 2. XÂY DỰNG MÔ HÌNH SVD VÀ CONTENT
# ==============================================
def build_svd_model(train_df, n_factors=110):
    users = train_df['user_id'].unique()
    items = train_df['parent_asin'].unique()
    user_to_idx = {user: idx for idx, user in enumerate(users)}   # user -> index
    item_to_idx = {item: idx for idx, item in enumerate(items)}   # item -> index
    n_users = len(users)
    n_items = len(items)
    interaction_matrix = np.zeros((n_users, n_items))
    for _, row in train_df.iterrows():
        u_idx = user_to_idx[row['user_id']]
        i_idx = item_to_idx[row['parent_asin']]
        interaction_matrix[u_idx, i_idx] = row['liked']
    
    k = min(n_factors, min(n_users, n_items) - 1)
    if k < 1:
        k = 1
    U, sigma, Vt = svds(interaction_matrix, k=k)
    sigma = np.diag(sigma)
    user_emb = U @ sigma
    item_emb = Vt.T
    
    user_emb_dict = {users[i]: user_emb[i] for i in range(n_users)}
    item_emb_dict = {items[i]: item_emb[i] for i in range(n_items)}
    return user_emb_dict, item_emb_dict, items

def build_content_model(train_df, X_train, other_features, use_nlp=True):
    item_to_rows = {}
    for idx, item in enumerate(train_df['parent_asin']):
        item_to_rows.setdefault(item, []).append(idx)
    
    all_items = train_df['parent_asin'].unique()
    item_vectors = []
    item_order = []
    
    for item in all_items:
        rows = item_to_rows[item]
        if use_nlp and X_train is not None:
            tfidf_vec = X_train[rows].mean(axis=0).A1
        else:
            tfidf_vec = np.array([])
        row = train_df[train_df['parent_asin'] == item].iloc[0]
        meta_vec = row[other_features].values.astype(np.float32)
        combined = np.concatenate([tfidf_vec, meta_vec]) if tfidf_vec.size else meta_vec
        item_vectors.append(combined)
        item_order.append(item)
    
    item_vectors = np.vstack(item_vectors)
    norms = np.linalg.norm(item_vectors, axis=1, keepdims=True)
    item_vectors = item_vectors / (norms + 1e-9)
    item_profiles = {item: item_vectors[i] for i, item in enumerate(item_order)}
    return item_profiles, item_order

def compute_user_content_profile(user_id, train_df, item_profiles, use_weighted=True, fallback_vector=None):
    liked_items = train_df[(train_df['user_id'] == user_id) & (train_df['liked'] == 1)]
    if len(liked_items) == 0:
        return fallback_vector if fallback_vector is not None else np.zeros(len(next(iter(item_profiles.values()))))
    
    vectors = []
    weights = []
    for _, row in liked_items.iterrows():
        item = row['parent_asin']
        if item in item_profiles:
            vectors.append(item_profiles[item])
            weights.append(1.0)  # có thể thay bằng rating gốc
    if not vectors:
        return fallback_vector if fallback_vector is not None else np.zeros(len(next(iter(item_profiles.values()))))
    
    vectors = np.array(vectors)
    weights = np.array(weights)
    if use_weighted:
        user_profile = np.average(vectors, axis=0, weights=weights)
    else:
        user_profile = np.mean(vectors, axis=0)
    norm = np.linalg.norm(user_profile)
    if norm > 0:
        user_profile /= norm
    return user_profile

# ==============================================
# 3. HÀM HYBRID RECOMMEND (USER-TO-ITEM VÀ ITEM-TO-ITEM)
# ==============================================
def hybrid_recommend_user(user_id, interacted_items, k, alpha,
                          user_emb, item_emb, user_content_profiles, item_profiles, all_items,
                          global_item_profile):
    if user_id in user_emb:
        user_collab_vec = user_emb[user_id]
    else:
        user_collab_vec = None
    user_content_vec = user_content_profiles.get(user_id, global_item_profile)
    
    scores = {}
    for item in all_items:
        if item in interacted_items:
            continue
        collab_score = 0
        if user_collab_vec is not None and item in item_emb:
            collab_score = np.dot(user_collab_vec, item_emb[item])
        content_score = np.dot(user_content_vec, item_profiles.get(item, np.zeros_like(user_content_vec)))
        scores[item] = alpha * collab_score + (1 - alpha) * content_score
    sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [item for item, _ in sorted_items[:k]]

def hybrid_recommend_item(item_id, k, alpha, item_emb, item_profiles, all_items):
    if item_id not in item_profiles and item_id not in item_emb:
        return []
    # Vector content và collaborative của item gốc
    content_vec = item_profiles.get(item_id, np.zeros(len(next(iter(item_profiles.values())))))
    collab_vec = item_emb.get(item_id, np.zeros(len(next(iter(item_emb.values()))))) if item_emb else np.zeros_like(content_vec)
    
    scores = {}
    for other in all_items:
        if other == item_id:
            continue
        other_content = item_profiles.get(other, np.zeros_like(content_vec))
        other_collab = item_emb.get(other, np.zeros_like(collab_vec))
        content_sim = np.dot(content_vec, other_content)
        collab_sim = np.dot(collab_vec, other_collab) if collab_vec.any() and other_collab.any() else 0
        scores[other] = alpha * collab_sim + (1 - alpha) * content_sim
    sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [item for item, _ in sorted_items[:k]]

# ==============================================
# 4. TÍCH HỢP VÀ THỬ NGHIỆM
# ==============================================
# Giả sử đã có train_df, test_df, X_train, other_features
other_features = other_features = [
    'brand_enc', 'store_enc', 'helpful_vote', 'average_rating', 'rating_number'
]  
user_emb, item_emb, all_items = build_svd_model(train_df, n_factors=110)
item_profiles, item_order = build_content_model(train_df, X_train, other_features, use_nlp=True)

global_item_profile = np.mean(list(item_profiles.values()), axis=0) if item_profiles else np.zeros(1)
user_content_profiles = {}
for user in train_df['user_id'].unique():
    user_content_profiles[user] = compute_user_content_profile(user, train_df, item_profiles,
                                                               use_weighted=True,
                                                               fallback_vector=global_item_profile)

# Thử nghiệm với các alpha khác nhau
alphas = [0.0, 0.3, 0.5, 0.7, 1.0]

print("=== Hybrid (User-to-item) ===")
for alpha in alphas:
    recommend_func = partial(hybrid_recommend_user, alpha=alpha,
                             user_emb=user_emb, item_emb=item_emb,
                             user_content_profiles=user_content_profiles,
                             item_profiles=item_profiles,
                             all_items=all_items,
                             global_item_profile=global_item_profile)
    results = evaluate_user_to_item(train_df, test_df, recommend_func, k=10)
    print(f"Alpha={alpha:.1f}: Precision={results['precision']:.4f}, Recall={results['recall']:.4f}, NDCG={results['ndcg']:.4f}")

print("\n=== Hybrid (Item-to-item) ===")
for alpha in alphas:
    recommend_item_func = partial(hybrid_recommend_item, alpha=alpha,
                                   item_emb=item_emb, item_profiles=item_profiles,
                                   all_items=all_items)
    item_gt = build_item_ground_truth(train_df, top_m=20)
    results = evaluate_item_to_item(train_df, item_gt, recommend_item_func, k=10)
    print(f"Alpha={alpha:.1f}: Precision={results['precision']:.4f}, Recall={results['recall']:.4f}, NDCG={results['ndcg']:.4f}")

=== Hybrid (User-to-item) ===
Alpha=0.0: Precision=0.0014, Recall=0.0126, NDCG=0.0062
Alpha=0.3: Precision=0.0014, Recall=0.0105, NDCG=0.0065
Alpha=0.5: Precision=0.0017, Recall=0.0136, NDCG=0.0067
Alpha=0.7: Precision=0.0019, Recall=0.0135, NDCG=0.0070
Alpha=1.0: Precision=0.0013, Recall=0.0097, NDCG=0.0052

=== Hybrid (Item-to-item) ===
Alpha=0.0: Precision=0.0442, Recall=0.0464, NDCG=0.0648
Alpha=0.3: Precision=0.0790, Recall=0.0698, NDCG=0.1054
Alpha=0.5: Precision=0.1412, Recall=0.1119, NDCG=0.1810
Alpha=0.7: Precision=0.2500, Recall=0.1898, NDCG=0.3114
Alpha=1.0: Precision=0.4809, Recall=0.4322, NDCG=0.5991
